# Imports & Config

In [126]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from math import nan

N_RECORDS = 20000
N_VEHICLES = 2500
START_YEAR = 2015
END_YEAR = 2025
SERVICE_INTERVAL_DAYS = 180

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
random.seed(42)

# Static Data

In [127]:
nissan_models = {
    "Sunny": {
        "engine": "I4",
        "engine_cc": 1500,
        "weight_kg": 1050,
        "vehicle_class": "Sedan",
        "drivetrain": "FWD",
        "base_failure_rate": 0.015
    },
    "Altima": {
        "engine": "I4",
        "engine_cc": 2500,
        "weight_kg": 1450,
        "vehicle_class": "Sedan",
        "drivetrain": "FWD",
        "base_failure_rate": 0.02
    },
    "X-Trail": {
        "engine": "I4",
        "engine_cc": 2500,
        "weight_kg": 1650,
        "vehicle_class": "SUV",
        "drivetrain": "AWD",
        "base_failure_rate": 0.025
    },
    "Pathfinder": {
        "engine": "V6",
        "engine_cc": 3500,
        "weight_kg": 2000,
        "vehicle_class": "SUV",
        "drivetrain": "AWD",
        "base_failure_rate": 0.032
    },
    "Patrol": {
        "engine": "V6",
        "engine_cc": 4000,
        "weight_kg": 2600,
        "vehicle_class": "SUV",
        "drivetrain": "AWD",
        "base_failure_rate": 0.035
    }
}

failure_types = [
    # Engine System
    "Engine Overheating",
    "Oil Leak",
    "Spark Plug Failure",
    "Fuel Injector Failure",

    # Brake System
    "Brake Pad Wear",
    "Brake Disc Warp",
    "Brake Fluid Leak",

    # Electrical System
    "Battery Failure",
    "Alternator Failure",
    "Starter Motor Failure",

    # Transmission System
    "Clutch Wear",
    "Gearbox Failure",
    "Transmission Fluid Leak",

    # Suspension & Steering
    "Shock Absorber Wear",
    "Control Arm Failure",
    "Wheel Alignment Issue",
    "Power Steering Failure",

    # Tires & Wheels
    "Tire Wear",
    "Tire Blowout",
    "Wheel Bearing Failure",

    # Cooling System
    "Radiator Leak",
    "Coolant Pump Failure",
    "Thermostat Failure",

    # AC System
    "AC Compressor Failure",
    "Refrigerant Leak"
]

parts_map = {
    # Engine
    "Engine Overheating": ["Radiator", "Coolant Pump", "Thermostat"],
    "Oil Leak": ["Oil Seal", "Gasket"],
    "Spark Plug Failure": ["Spark Plug"],
    "Fuel Injector Failure": ["Fuel Injector"],

    # Brake
    "Brake Pad Wear": ["Brake Pads"],
    "Brake Disc Warp": ["Brake Disc"],
    "Brake Fluid Leak": ["Brake Hose", "Brake Fluid"],

    # Electrical
    "Battery Failure": ["Battery"],
    "Alternator Failure": ["Alternator"],
    "Starter Motor Failure": ["Starter Motor"],

    # Transmission
    "Clutch Wear": ["Clutch Kit"],
    "Gearbox Failure": ["Gearbox"],
    "Transmission Fluid Leak": ["Transmission Seal", "Transmission Fluid"],

    # Suspension
    "Shock Absorber Wear": ["Shock Absorber"],
    "Control Arm Failure": ["Control Arm"],
    "Wheel Alignment Issue": ["Alignment Service"],
    "Power Steering Failure": ["Power Steering Pump"],

    # Tires
    "Tire Wear": ["Tire"],
    "Tire Blowout": ["Tire"],
    "Wheel Bearing Failure": ["Wheel Bearing"],

    # Cooling
    "Radiator Leak": ["Radiator"],
    "Coolant Pump Failure": ["Coolant Pump"],
    "Thermostat Failure": ["Thermostat"],

    # AC
    "AC Compressor Failure": ["AC Compressor"],
    "Refrigerant Leak": ["Refrigerant"]
}

parts_catalog = {

    # -------------------------
    # Engine
    # -------------------------
    "Radiator": {
        "cost": 75,
        "labor_hours": 2.0
    },

    "Coolant Pump": {
        "cost": 45,
        "labor_hours": 2.0
    },

    "Thermostat": {
        "cost": 18,
        "labor_hours": 1.0
    },

    "Oil Seal": {
        "cost": 8,
        "labor_hours": 2.0
    },

    "Gasket": {
        "cost": 12,
        "labor_hours": 2.0
    },

    "Spark Plug": {
        "cost": 8,
        "labor_hours": 0.5
    },

    "Fuel Injector": {
        "cost": 35,
        "labor_hours": 1.5
    },

    # -------------------------
    # Brakes
    # -------------------------
    "Brake Pads": {
        "cost": 32,
        "labor_hours": 1.0
    },

    "Brake Disc": {
        "cost": 30,
        "labor_hours": 1.5
    },

    "Brake Hose": {
        "cost": 12,
        "labor_hours": 1.0
    },

    "Brake Fluid": {
        "cost": 4,
        "labor_hours": 0.5
    },

    # -------------------------
    # Electrical
    # -------------------------
    "Battery": {
        "cost": 45,
        "labor_hours": 0.5
    },

    "Alternator": {
        "cost": 85,
        "labor_hours": 2.0
    },

    "Starter Motor": {
        "cost": 70,
        "labor_hours": 2.0
    },

    # -------------------------
    # Transmission
    # -------------------------
    "Clutch Kit": {
        "cost": 140,
        "labor_hours": 4.0
    },

    "Gearbox": {
        "cost": 600,
        "labor_hours": 6.0
    },

    "Transmission Seal": {
        "cost": 15,
        "labor_hours": 2.0
    },

    "Transmission Fluid": {
        "cost": 25,
        "labor_hours": 1.0
    },

    # -------------------------
    # Suspension / Steering
    # -------------------------
    "Shock Absorber": {
        "cost": 55,
        "labor_hours": 2.0
    },

    "Control Arm": {
        "cost": 60,
        "labor_hours": 2.0
    },

    "Alignment Service": {
        "cost": 8,
        "labor_hours": 1.0
    },

    "Power Steering Pump": {
        "cost": 100,
        "labor_hours": 3.0
    },

    # -------------------------
    # Tires / Wheels
    # -------------------------
    "Tire": {
        "cost": 45,
        "labor_hours": 0.5
    },

    "Wheel Bearing": {
        "cost": 35,
        "labor_hours": 2.0
    },

    # -------------------------
    # Cooling
    # -------------------------
    "AC Compressor": {
        "cost": 180,
        "labor_hours": 3.0
    },

    "Refrigerant": {
        "cost": 15,
        "labor_hours": 1.0
    }
}

failure_weights = {
    # High-frequency wear items
    "Brake Pad Wear": 0.12,
    "Tire Wear": 0.10,
    "Shock Absorber Wear": 0.08,
    "Clutch Wear": 0.07,

    # Medium frequency
    "Battery Failure": 0.08,
    "Oil Leak": 0.07,
    "Wheel Alignment Issue": 0.06,
    "Brake Disc Warp": 0.05,
    "Control Arm Failure": 0.05,

    # Lower frequency but common faults
    "Spark Plug Failure": 0.05,
    "Fuel Injector Failure": 0.04,
    "Alternator Failure": 0.04,
    "Radiator Leak": 0.04,

    # Rare but impactful
    "Engine Overheating": 0.03,
    "Gearbox Failure": 0.02,
    "Starter Motor Failure": 0.02,
    "Power Steering Failure": 0.02,
    "Wheel Bearing Failure": 0.02,

    # Environment-driven
    "Tire Blowout": 0.02,
    "Coolant Pump Failure": 0.02,
    "Thermostat Failure": 0.02,

    # AC-related
    "AC Compressor Failure": 0.02,
    "Refrigerant Leak": 0.02,

    # Rare leaks
    "Brake Fluid Leak": 0.015,
    "Transmission Fluid Leak": 0.015
}

failure_severity_map = {
    # Critical = Emergency likely
    "Engine Overheating": "High",
    "Gearbox Failure": "High",
    "Brake Fluid Leak": "High",
    "Tire Blowout": "High",
    "Power Steering Failure": "High",

    # Medium = Corrective mostly
    "Alternator Failure": "Medium",
    "Starter Motor Failure": "Medium",
    "Radiator Leak": "Medium",
    "Coolant Pump Failure": "Medium",
    "AC Compressor Failure": "Medium",
    "Control Arm Failure": "Medium",

    # Low = Planned repair
    "Brake Pad Wear": "Low",
    "Tire Wear": "Low",
    "Clutch Wear": "Low",
    "Oil Leak": "Low",
    "Spark Plug Failure": "Low",
    "Wheel Alignment Issue": "Low"
}

seasons = ["Winter", "Spring", "Summer", "Autumn"]

# Helper Functions

In [128]:
def random_date(start_year, end_year):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))


def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


def temperature_by_season(season):
    if season == "Summer":
        return np.random.normal(42, 5)
    elif season == "Winter":
        return np.random.normal(22, 4)
    elif season == "Spring":
        return np.random.normal(30, 4)
    else:
        return np.random.normal(32, 4)

def get_weighted_failure(
    failure_weights,
    mileage,
    temp,
    vehicle_age,
    missed_services,
    driving_style,
    engine_cc,
    weight_kg,
    vehicle_class,
    drivetrain
):
    adjusted_weights = failure_weights.copy()

    for failure in adjusted_weights:

        # ---------------------------
        # 1. HEAT (Oman realism)
        # ---------------------------
        if temp > 40:
            if "Battery" in failure:
                adjusted_weights[failure] *= 1.8
            if "Overheating" in failure or "Radiator" in failure:
                adjusted_weights[failure] *= 1.6
            if "Tire" in failure:
                adjusted_weights[failure] *= 1.5

        # Engine size amplifies heat problems
        if engine_cc > 3000 and temp > 38:
            if "Overheating" in failure:
                adjusted_weights[failure] *= 1.5

        # ---------------------------
        # 2. WEIGHT PHYSICS
        # ---------------------------
        if weight_kg > 1800:
            if "Suspension" in failure or "Shock" in failure or "Control Arm" in failure:
                adjusted_weights[failure] *= 1.6
            if "Brake" in failure:
                adjusted_weights[failure] *= 1.4
            if "Tire" in failure:
                adjusted_weights[failure] *= 1.3

        elif weight_kg < 1200:
            if "Suspension" in failure or "Shock" in failure or "Control Arm" in failure:
                adjusted_weights[failure] *= 0.7
            if "Brake" in failure:
                adjusted_weights[failure] *= 0.8

        # ---------------------------
        # 3. VEHICLE CLASS EFFECTS
        # ---------------------------
        if vehicle_class == "SUV":
            if "Suspension" in failure or "Shock" in failure or "Control Arm" in failure:
                adjusted_weights[failure] *= 1.4
            if "Alignment" in failure:
                adjusted_weights[failure] *= 1.3

        # ---------------------------
        # 4. DRIVETRAIN COMPLEXITY
        # ---------------------------
        if drivetrain == "AWD":
            if "Transmission" in failure or "Differential" in failure:
                adjusted_weights[failure] *= 1.4

        # ---------------------------
        # 5. MILEAGE WEAR
        # ---------------------------
        if mileage > 100000:
            if any(x in failure for x in ["Wear", "Brake", "Clutch", "Bearing"]):
                adjusted_weights[failure] *= 1.5

        if mileage > 180000:
            if "Engine" in failure or "Transmission" in failure:
                adjusted_weights[failure] *= 1.6

        # ---------------------------
        # 6. AGE DEGRADATION
        # ---------------------------
        if vehicle_age > 8:
            if "Leak" in failure or "Seal" in failure:
                adjusted_weights[failure] *= 1.5

        if vehicle_age > 12:
            if "Electrical" in failure or "Sensor" in failure:
                adjusted_weights[failure] *= 1.4

        # ---------------------------
        # 7. MISSED SERVICE CASCADE
        # ---------------------------
        if missed_services > 0:
            if "Oil" in failure or "Engine" in failure:
                adjusted_weights[failure] *= 1.6

        if missed_services > 2:
            if "Engine" in failure:
                adjusted_weights[failure] *= 1.8

        # ---------------------------
        # 8. DRIVING STYLE
        # ---------------------------
        if driving_style == "Aggressive":
            if any(x in failure for x in ["Brake", "Clutch", "Tire"]):
                adjusted_weights[failure] *= 1.5
            if "Transmission" in failure:
                adjusted_weights[failure] *= 1.3

        # ---------------------------
        # 9. INTERACTION EFFECTS 
        # ---------------------------

        # Heavy SUV + aggressive = suspension killer
        if weight_kg > 2000 and driving_style == "Aggressive":
            if "Suspension" in failure or "Shock" in failure or "Control Arm" in failure:
                adjusted_weights[failure] *= 1.7

        # High mileage + missed service = catastrophic engine
        if mileage > 150000 and missed_services > 1:
            if "Engine" in failure:
                adjusted_weights[failure] *= 2.0

        # Heat + old vehicle = cooling system collapse
        if temp > 40 and vehicle_age > 10:
            if "Radiator" in failure or "Cooling" in failure:
                adjusted_weights[failure] *= 1.7

    # ---------------------------
    # Normalize + sample
    # ---------------------------
    total = sum(adjusted_weights.values())
    probs = [w / total for w in adjusted_weights.values()]

    return np.random.choice(list(adjusted_weights.keys()), p=probs)

def determine_service_type(failure_type, severity_map, temp, mileage):
    
    if failure_type == "None":
        return "Preventive"

    severity = severity_map.get(failure_type, "Medium")

    # High severity = mostly emergency
    if severity == "High":
        return np.random.choice(
            ["Emergency", "Corrective"],
            p=[0.8, 0.2]
        )

    # Medium severity = depends on conditions
    elif severity == "Medium":
        emergency_prob = 0.3

        # Heat increases urgency
        if temp > 40:
            emergency_prob += 0.2

        # Very high mileage = more catastrophic behavior
        if mileage > 150000:
            emergency_prob += 0.1

        emergency_prob = min(emergency_prob, 0.99)

        return np.random.choice(
            ["Emergency", "Corrective"],
            p=[emergency_prob, 1 - emergency_prob]
        )

    # Low severity = almost always corrective
    else:
        return np.random.choice(
            ["Corrective", "Emergency"],
            p=[0.99, 0.01]
        )
    
def calculate_service_cost(parts, vehicle_class, labor_rate_omr=8):
    total_parts_cost = 0
    total_labor_hours = 0

    for part in parts:
        part_info = parts_catalog.get(
            part,
            {"cost": 10, "labor_hours": 1} # Fallback
        )

        total_parts_cost += part_info["cost"]
        total_labor_hours += part_info["labor_hours"]

    # SUV labor adjustment
    if vehicle_class == "SUV":
        total_labor_hours *= 1.2

    labor_cost = total_labor_hours * labor_rate_omr

    # Parts + labor
    total_cost = total_parts_cost + labor_cost

    # ±10% realistic price variation
    total_cost = round(total_cost * np.random.uniform(0.90, 1.10),1)

    return round(total_cost, 2)

# Vehicle Registry

In [129]:
vehicle_registry = {}

for vid in range(1, N_VEHICLES + 1):

    vehicle_id = f"V{vid}"

    # -------------------------
    # Select vehicle model
    # -------------------------
    model = random.choice(list(nissan_models.keys()))
    model_data = nissan_models[model]

    # -------------------------
    # Vehicle specifications
    # -------------------------
    engine = model_data["engine"]
    engine_cc = model_data["engine_cc"]
    weight_kg = model_data["weight_kg"]
    vehicle_class = model_data["vehicle_class"]
    drivetrain = model_data["drivetrain"]
    base_failure_rate = model_data["base_failure_rate"]

    # -------------------------
    # Vehicle age
    # -------------------------
    manufacture_year = random.randint(2008, 2022)

    # -------------------------
    # Driving behavior
    # -------------------------
    driving_style = random.choices(
        ["Calm", "Aggressive"],
        weights=[0.80, 0.20]
    )[0]

    # Maintenance diligence
    if driving_style == "Calm":
        diligence = random.choices(
            ["Diligent", "Irresponsible"],
            weights=[0.70, 0.30]
        )[0]
    else:
        diligence = random.choices(
            ["Diligent", "Irresponsible"],
            weights=[0.65, 0.35]
        )[0]

    # -------------------------
    # Daily usage
    # -------------------------
    avg_daily_km = round(max(
        5,
        np.random.normal(40, 10)
    ), 1)

    vehicle_registry[vehicle_id] = {
        # Identification
        "model": model,

        # Vehicle specifications
        "engine": engine,
        "engine_cc": engine_cc,
        "weight_kg": weight_kg,
        "vehicle_class": vehicle_class,
        "drivetrain": drivetrain,

        # Reliability
        "base_failure_rate": base_failure_rate,

        # Vehicle lifecycle
        "manufacture_year": manufacture_year,

        # Usage / behavior
        "driving_style": driving_style,
        "diligence": diligence,
        "avg_daily_km": avg_daily_km
    }

# Generator

In [130]:
records = []

for vehicle_id, v in vehicle_registry.items():

    current_date = datetime(v["manufacture_year"], 1, 1)
    end_date = datetime(END_YEAR, 12, 31)

    mileage = 0
    last_service_date = current_date
    missed_services = 0

    while current_date < end_date:
        service_intreval = int(np.random.normal(SERVICE_INTERVAL_DAYS, 15)) # Service delays (Noise)

        # Move to next scheduled service
        next_service_date = last_service_date + timedelta(days=service_intreval)

        # Advance time
        days_passed = (next_service_date - current_date).days
        mileage += int(v["avg_daily_km"] * days_passed)

        current_date = next_service_date

        season = get_season(current_date)
        temp = round(temperature_by_season(season), 1)

        vehicle_age = current_date.year - v["manufacture_year"]

        # Did the vehicle actually come in for service?
        if v["diligence"] == "Diligent":
            serviced = np.random.rand() > 0.01 
        else:
            serviced = np.random.rand() > 0.33   

        if not serviced:
            missed_services += 1

        # Failure probability model 
        failure_prob = v["base_failure_rate"]
        failure_prob += vehicle_age * 0.005
        failure_prob += mileage / 200000
        failure_prob += missed_services * 0.20  # penalty for skipping

        if season == "Summer":
            failure_prob += 0.05

        if v["driving_style"] == "Aggressive":
            failure_prob += 0.05

        failure_prob = min(failure_prob, 0.99)

        failure_occurred = np.random.rand() < failure_prob

        if failure_occurred:
            failure_type = get_weighted_failure(
                failure_weights,
                mileage,
                temp,
                vehicle_age,
                missed_services,
                v["driving_style"],
                v["engine_cc"],
                v["weight_kg"],
                v["vehicle_class"],
                v["drivetrain"]
            )
            service_type = determine_service_type(
                failure_type,
                failure_severity_map,
                temp,
                mileage
            )
            parts = parts_map[failure_type]
            cost = calculate_service_cost(parts, v["vehicle_class"])

            # Reset after failure repair
            missed_services = 0
            last_service_date = current_date

        elif serviced:
            failure_type = "None"
            service_type = "Preventive"
            parts = ["Oil Filter", "Engine Oil"]

            # Regular preventive service (OMR)
            cost = round(np.random.uniform(15, 30), 1)

            # Successful service resets missed count
            missed_services = 0
            last_service_date = current_date

        else:
            # Skipped service
            failure_type = "None"
            service_type = nan
            parts = ""
            cost = nan
            service_type = "Skipped"
            cost = 0

        records.append({
            "vehicle_id": vehicle_id,
            "model": v["model"],
            "engine_type": v["engine"],
            "manufacture_year": v["manufacture_year"],
            "service_date": current_date,
            "season": season,
            "ambient_temp": temp,
            "vehicle_age": vehicle_age,
            "mileage": mileage,
            "avg_daily_km": v["avg_daily_km"],
            "driving_style": v["driving_style"],
            "diligence": v["diligence"],
            "missed_services": missed_services,
            "service_type": service_type,
            "failure_occurred": int(failure_occurred),
            "failure_type": failure_type,
            "parts_replaced": ",".join(parts),
            "service_cost": cost,
        })

df = pd.DataFrame(records)

# Save

In [131]:
df.to_csv("synthetic_nissan_pdm.csv", index=False)

# Analysis

In [ ]:
df = pd.read_csv("synthetic_nissan_pdm.csv")

In [ ]:
df[(df["vehicle_id"].duplicated(keep=False)) & (df["vehicle_id"] == "V457")]

In [ ]:
df[(df["service_type"]=="Emergency")]

In [ ]:
df[(df["service_type"]=="Preventive")]

In [ ]:
# identify column types
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Numerical columns ({len(num_cols)}):", num_cols)
print(f"Categorical columns ({len(cat_cols)}):", cat_cols)

In [ ]:
for col in num_cols:
    fig, axes = plt.subplots(1, 1, figsize=(8, 5))
    
    # Histogram + KDE
    sns.histplot(df[col].dropna(), kde=True, ax=axes)
    axes.set_title(f"{col} Distribution")
    
    # # Boxplot
    # sns.boxplot(x=df[col], ax=axes[1])
    # axes[1].set_title(f"{col} Boxplot")
    
    plt.tight_layout()
    plt.show()

In [ ]:
for col in cat_cols:
    plt.figure(figsize=(10, 5))
    
    # limit categories for readability
    top_vals = df[col].value_counts().nlargest(20)
    
    sns.barplot(x=top_vals.values, y=top_vals.index)
    plt.title(f"{col} Top Categories")
    
    plt.tight_layout()
    plt.show()

In [133]:
# Failure type by car model
failure_model = (
    df[df["failure_occurred"] == 1]
    .groupby(["model", "failure_type"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    failure_model,
    x="model",
    y="count",
    color="failure_type",
    title="Failure Type by Vehicle Model",
    labels={
        "model": "Vehicle Model",
        "count": "Number of Failures",
        "failure_type": "Failure Type"
    },
    hover_data={
        "model": True,
        "failure_type": True,
        "count": True
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    hovermode="closest"
)

fig.show(renderer = 'browser')

In [ ]:
# Failures over mileage 
failure_mileage = df[df["failure_occurred"] == 1].copy()

# Create 10,000 km mileage bins
failure_mileage["mileage_start"] = (
    (failure_mileage["mileage"] // 10000) * 10000
)

failure_mileage["mileage_range"] = (
    failure_mileage["mileage_start"].astype(int).astype(str)
    + "–"
    + (failure_mileage["mileage_start"] + 9999).astype(int).astype(str)
    + " km"
)

# Count failures by mileage bin and failure type
failure_mileage_grouped = (
    failure_mileage
    .groupby(["mileage_start", "mileage_range", "failure_type"])
    .size()
    .reset_index(name="count")
)

# Rank failure types within each mileage bin
failure_mileage_grouped["rank"] = (
    failure_mileage_grouped
    .groupby("mileage_start")["count"]
    .rank(method="first", ascending=False)
)

# Keep top 5 failures per mileage bin
top5_failures = (
    failure_mileage_grouped[
        failure_mileage_grouped["rank"] <= 5
    ]
    .sort_values(["mileage_start", "count"], ascending=[True, False])
)

fig = px.bar(
    top5_failures,
    x="mileage_range",
    y="count",
    color="failure_type",
    title="Top 5 Failure Types Across Mileage Ranges",
    labels={
        "mileage_range": "Mileage",
        "count": "Number of Failures",
        "failure_type": "Failure Type"
    },
    hover_data={
        "failure_type": True,
        "count": True
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    hovermode="closest"
)

fig.show(renderer="browser")

In [132]:
## Failures by car type
# Map model to vehicle class
model_class = {
    model: info["vehicle_class"]
    for model, info in nissan_models.items()
}

df["vehicle_class"] = df["model"].map(model_class)

# Only failures
failures = df[df["failure_occurred"] == 1]

# Count failures by model + failure type
failure_counts = (
    failures
    .groupby(["model", "vehicle_class", "failure_type"])
    .size()
    .reset_index(name="failures")
)

# Count ALL records by model
total_records = (
    df
    .groupby(["model", "vehicle_class"])
    .size()
    .reset_index(name="total_records")
)

# Calculate failure rate for each model
model_failure_rate = failure_counts.merge(
    total_records,
    on=["model", "vehicle_class"]
)

model_failure_rate["failure_rate"] = (
    model_failure_rate["failures"]
    / model_failure_rate["total_records"]
    * 100
)

# Average the model-level rates within each vehicle class
class_failure_rate = (
    model_failure_rate
    .groupby(["vehicle_class", "failure_type"])["failure_rate"]
    .mean()
    .reset_index()
)

fig = px.bar(
    class_failure_rate,
    x="failure_type",
    y="failure_rate",
    color="vehicle_class",
    barmode="group",
    title="Model-Balanced Failure Rates: Sedan vs SUV",
    labels={
        "failure_type": "Failure Type",
        "failure_rate": "Average Failure Rate (%)",
        "vehicle_class": "Vehicle Class"
    },
    hover_data={
        "vehicle_class": True,
        "failure_type": True,
        "failure_rate": ":.2f"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    hovermode="closest"
)

fig.show(renderer="browser")

In [ ]:
# Failure count by driving style
driving_failure_rate = (
    df.groupby("driving_style")["failure_occurred"]
    .agg(["sum", "count"])
    .reset_index()
)

driving_failure_rate["failure_rate"] = (
    driving_failure_rate["sum"]
    / driving_failure_rate["count"]
    * 100
)

driving_failure_rate = driving_failure_rate.rename(
    columns={
        "sum": "failures",
        "count": "total_records"
    }
)

fig = px.bar(
    driving_failure_rate,
    x="driving_style",
    y="failure_rate",
    color="driving_style",
    title="Normalized Failure Rate by Driving Style",
    labels={
        "driving_style": "Driving Style",
        "failure_rate": "Failure Rate (%)"
    },
    hover_data={
        "driving_style": True,
        "failures": True,
        "total_records": True,
        "failure_rate": ":.2f"
    }
)

fig.update_layout(
    showlegend=False,
    hovermode="closest"
)

fig.show(renderer="browser")

In [ ]:
# Failure count by diligence
diligence_failure_rate = (
    df.groupby("diligence")["failure_occurred"]
    .agg(["sum", "count"])
    .reset_index()
)

diligence_failure_rate["failure_rate"] = (
    diligence_failure_rate["sum"]
    / diligence_failure_rate["count"]
    * 100
)

diligence_failure_rate = diligence_failure_rate.rename(
    columns={
        "sum": "failures",
        "count": "total_records"
    }
)

fig = px.bar(
    diligence_failure_rate,
    x="diligence",
    y="failure_rate",
    color="diligence",
    title="Normalized Failure Rate by Diligence",
    labels={
        "diligence": "Diligence",
        "failure_rate": "Failure Rate (%)"
    },
    hover_data={
        "diligence": True,
        "failures": True,
        "total_records": True,
        "failure_rate": ":.2f"
    }
)

fig.update_layout(
    showlegend=False,
    hovermode="closest"
)

fig.show(renderer="browser")